In [ ]:
num_epochs = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

You train the model for 10 full passes over the training dataset.

These lists store metrics for each epoch so you can later:

* Plot learning curves

* Analyze overfitting

* Compare performance trends

In [ ]:
for epoch in range(num_epochs):

    # -------- TRAINING --------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)


    # -------- VALIDATION --------
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)


    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}% "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

 ``` model.train() ```

This sets the model to training mode.

Important because:

* Dropout is activated

* BatchNorm updates running statistics

* Some layers behave differently during training vs evaluation
---

running_loss → accumulates batch losses

correct → counts correct predictions

total → counts total samples seen

---

``` optimizer.zero_grad()```

PyTorch accumulates gradients by default.
So we reset them before computing new gradients.

---

``` outputs = model(images) ```

The model:

Takes images as input

Produces raw outputs (logits)

---
``` loss = criterion(outputs, labels) ```

criterion is typically:

nn.CrossEntropyLoss() for classification

Loss measures how wrong predictions are.

---

loss.backward()

This computes gradients:
```
∂Loss/​∂Parameters
```
Using backpropagation.

---
optimizer.step()

Updates model parameters using gradients.

Example (if SGD):

* w=w−η⋅∇w
---

```running_loss += loss.item()```

loss.item() extracts scalar value

Adds batch loss to total

---
```_, predicted = torch.max(outputs.data, 1)```

torch.max(..., 1) finds class with highest score

Returns:

* value (ignored _)

* index (predicted class)

---
```
total += labels.size(0)
correct += (predicted == labels).sum().item()
```
Count total samples

Count how many predictions are correct

---
```
train_loss = running_loss / len(train_loader)
```
Average loss per batch.
```
train_accuracy = 100 * correct / total
```
Percentage accuracy.

Then store them for plotting later.

---
> model.eval()

Disables dropout

Freezes BatchNorm statistics

Ensures deterministic behavior

---

```with torch.no_grad():```

* Saves memory

* Speeds up validation

* No gradient computation needed

---
```
val_loss = val_running_loss / len(val_loader)
val_accuracy = 100 * val_correct / val_total
```

These measure generalization performance.

In [ ]:
model.eval()
with torch.no_grad():
  plt.figure(figsize=(12,5))

  plt.subplot(1,2,1)
  plt.plot(train_losses, label="Train Loss")
  plt.plot(val_losses, label="Validation Loss")
  plt.legend()
  plt.title("Loss vs Epoch")

  plt.subplot(1,2,2)
  plt.plot(train_accuracies, label="Train Accuracy")
  plt.plot(val_accuracies, label="Validation Accuracy")
  plt.legend()
  plt.title("Accuracy vs Epoch")

  plt.show()